![image info](https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2023/main/images/banner_1.png)

# Proyecto 1 - Predicción de popularidad en canción

En este proyecto podrán poner en práctica sus conocimientos sobre modelos predictivos basados en árboles y ensambles, y sobre la disponibilización de modelos. Para su desarrollo tengan en cuenta las instrucciones dadas en la "Guía del proyecto 1: Predicción de popularidad en canción".

**Entrega**: La entrega del proyecto deberán realizarla durante la semana 4. Sin embargo, es importante que avancen en la semana 3 en el modelado del problema y en parte del informe, tal y como se les indicó en la guía.

Para hacer la entrega, deberán adjuntar el informe autocontenido en PDF a la actividad de entrega del proyecto que encontrarán en la semana 4, y subir el archivo de predicciones a la competencia de Kaggle cuyo link estará disponible en la sección del Coursera del proyecto.

## Datos para la predicción de popularidad en cancion

En este proyecto se usará el conjunto de datos de datos de popularidad en canciones, donde cada observación representa una canción y se tienen variables como: duración de la canción, acusticidad y tempo, entre otras. El objetivo es predecir qué tan popular es la canción. Para más detalles puede visitar el siguiente enlace: [datos](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset).

## Ejemplo predicción conjunto de test para envío a Kaggle

En esta sección encontrarán el formato en el que deben guardar los resultados de la predicción para que puedan subirlos a la competencia en Kaggle.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Importación librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor, Pool

In [ ]:
# Carga de datos de archivo .csv
train = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTrain_Spotify.csv')
test = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2026/main/datasets/dataTest_Spotify.csv', index_col=0)

target = "popularity"
drop_cols = ["Unnamed: 0", "track_id"]

X = train.drop(columns=[target] + drop_cols).copy()
y = train[target].copy()
X_test = test.drop(columns=["track_id"]).copy()

In [ ]:
# Nulos en texto
for df in [X, X_test]:
    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].fillna("missing")

In [ ]:
def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
# Features extra
for df in [X, X_test]:
    df["duration_min"] = df["duration_ms"] / 60000.0
    df["track_name_len"] = df["track_name"].astype(str).str.len()
    df["album_name_len"] = df["album_name"].astype(str).str.len()
    df["artists_len"] = df["artists"].astype(str).str.len()
    df["n_artists"] = df["artists"].astype(str).str.count(",") + 1

cat_cols = ["artists", "album_name", "track_name", "track_genre", "explicit", "key", "mode", "time_signature"]
cat_idx = [X.columns.get_loc(c) for c in cat_cols]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(X))
test_pred = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
    valid_pool = Pool(X_va, y_va, cat_features=cat_idx)

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        iterations=3000,
        learning_rate=0.05,
        depth=8,
        random_seed=42,
        od_type="Iter",
        od_wait=150,
        verbose=200
    )

    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
    pred_va = model.predict(X_va)
    pred_va = np.clip(pred_va, 0, 100)
    oof[va_idx] = pred_va

    rmse = root_mean_squared_error(y_va, pred_va)
    print(f"Fold {fold} RMSE: {rmse:.4f}")

    test_pred += model.predict(X_test) / kf.n_splits

final_pred = np.clip(test_pred, 0, 100)

submission = pd.DataFrame({
    "ID": test["Unnamed: 0"],
    "Popularity": final_pred
})
submission.to_csv('test_submission_file.csv', index_label='ID')
submission.head()

In [ ]:
import pandas as pd

revision = pd.read_csv('test_submission_file.csv')
df = revision.loc[:, ~revision.columns.duplicated()]
df = df[["ID", "Popularity"]]
df.to_csv("test_submission_file_V1.csv", index=False)

In [ ]:
df.head(15)